# Pengujian Forward Propagation CNN from Scratch 

#### Import Dataset

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import numpy as np

# impor modul dari folder src/model
import sys
import os
sys.path.append(os.path.abspath("../../src"))
from model.Model import Model

Seeding

In [2]:
import random
import os

# Atur seed
seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)
tf.keras.utils.set_random_seed(seed_value)
tf.config.experimental.enable_op_determinism()
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['PYTHONHASHSEED'] = '42'

#### Load Dataset

In [3]:
# Load dataset CIFAR-10
(x_train_full, y_train_full), (x_test, y_test) = cifar10.load_data()

# Split train -> 40k train, 10k validation
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full)

# Normalisasi data (0-1)
x_train = x_train.astype('float32') / 255.0
x_val = x_val.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

print(f"Train shape: {x_train.shape}, Validation shape: {x_val.shape}, Test shape: {x_test.shape}")

Train shape: (40000, 32, 32, 3), Validation shape: (10000, 32, 32, 3), Test shape: (10000, 32, 32, 3)


## Test 1

### Model Keras

In [ ]:
# Make model with 1 convolutional layer
model = models.Sequential([
    layers.Conv2D(8, (3, 3), activation='relu', input_shape=(32, 32, 3)),       # Conv2D layer
    layers.MaxPooling2D((2, 2)),                                                # Pooling layers
    layers.Flatten(),                                                           # Flatten
    layers.Dense(128, activation='relu'),                                       # Dense layer
    layers.Dense(10, activation='softmax')
])

# Compile model
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

# Callback
early_stop = tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

# Training model
history = model.fit(
    x_train, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(x_val, y_val),
    callbacks=[early_stop]
)

# Prediksi di test set
y_pred_probs = model.predict(x_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# Hitung macro F1-Score
f1 = f1_score(y_test, y_pred, average='macro')
print(f"Macro F1-Score di Test Set: {f1:.4f}")

# save model
model.save('cnn_model1.h5')

c:\Users\agilf\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.3422 - loss: 1.8213 - val_accuracy: 0.4910 - val_loss: 1.4369
Epoch 2/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.5166 - loss: 1.3756 - val_accuracy: 0.5321 - val_loss: 1.3317
Epoch 3/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.5563 - loss: 1.2566 - val_accuracy: 0.5506 - val_loss: 1.2734
Epoch 4/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.5872 - loss: 1.1777 - val_accuracy: 0.5684 - val_loss: 1.2288
Epoch 5/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.6091 - loss: 1.1144 - val_accuracy: 0.5777 - val_loss: 1.2031
Epoch 6/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.6286 - loss: 1.0620 - val_accuracy: 0.5879 - val_loss: 1.1917
Epoch 7/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.6459 - loss: 1.0138 - val_accuracy: 0.5933 - val_loss: 1.1822
Epoch 8/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.6623 - loss: 0.9705 - val_accuracy: 0

Macro F1-Score di Test Set: 0.5824


### Model Scratch

In [5]:
my_model = Model()
my_model.set_input_shape((32, 32, 3))
my_model.load('cnn_model1.h5')

# Prediksi di test set
y_pred_probs = my_model.predict(x_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# Hitung macro F1-Score
f1 = f1_score(y_test, y_pred, average='macro')
print(f"Macro F1-Score di Test Set: {f1:.4f}")

Processing layer 0: Conv2D
data processed: 10000/10000
Processing layer 0: MaxPooling
data processed: 10000/10000
Processing layer 0: Flatten
Processing layer 0: Dense
Processing layer 0: Dense
Macro F1-Score di Test Set: 0.5824


## Test 2

### Model Keras

In [ ]:
# Make model with 1 convolutional layer
model = models.Sequential([
    layers.Conv2D(2, (3, 3), activation='relu', input_shape=(32, 32, 3)),   # Conv2D layer
    layers.MaxPooling2D((2, 2)),                                            # Pooling layers
    layers.Conv2D(2, (3, 3), activation='relu'),
    layers.AveragePooling2D((2, 2)),
    layers.GlobalAveragePooling2D(),                                        # Global Average Pooling
    layers.Dense(128, activation='relu'),                                   # Dense layer
    layers.Dense(10, activation='softmax')
])

# Compile model
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

# Callback
early_stop = tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

# Training model
history = model.fit(
    x_train, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(x_val, y_val),
    callbacks=[early_stop]
)

# Prediksi di test set
y_pred_probs = model.predict(x_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# Hitung macro F1-Score
f1 = f1_score(y_test, y_pred, average='macro')
print(f"Macro F1-Score di Test Set: {f1:.4f}")

# save model
model.save('cnn_model2.h5')

c:\Users\agilf\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.1469 - loss: 2.2139 - val_accuracy: 0.2004 - val_loss: 2.0665
Epoch 2/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.2082 - loss: 2.0537 - val_accuracy: 0.2143 - val_loss: 2.0334
Epoch 3/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.2209 - loss: 2.0238 - val_accuracy: 0.2260 - val_loss: 2.0089
Epoch 4/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.2312 - loss: 2.0042 - val_accuracy: 0.2292 - val_loss: 1.9951
Epoch 5/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.2353 - loss: 1.9914 - val_accuracy: 0.2304 - val_loss: 1.9830
Epoch 6/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.2383 - loss: 1.9798 - val_accuracy: 0.2352 - val_loss: 1.9719
Epoch 7/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.2438 - loss: 1.9687 - val_accuracy: 0.2422 - val_loss: 1.9622
Epoch 8/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.2469 - loss: 1.9586 - val_accuracy: 0.

Macro F1-Score di Test Set: 0.2492


In [7]:
my_model = Model()
my_model.set_input_shape((32, 32, 3))
my_model.load('cnn_model2.h5')

# Prediksi di test set
y_pred_probs = my_model.predict(x_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# Hitung macro F1-Score
f1 = f1_score(y_test, y_pred, average='macro')
print(f"Macro F1-Score di Test Set: {f1:.4f}")

Processing layer 0: Conv2D
data processed: 10000/10000
Processing layer 0: MaxPooling
data processed: 10000/10000
Processing layer 0: Conv2D
data processed: 10000/10000
Processing layer 0: AveragePooling
data processed: 10000/10000
Processing layer 0: GlobalAveragePooling
Processing layer 0: Dense
Processing layer 0: Dense
Macro F1-Score di Test Set: 0.2492
